# BBO Capstone: round walkthrough

This notebook runs one round of the Black-Box Optimisation capstone step by step, so each decision is visible.

**Each round:**
1. Log the previous round's real portal results in `Data/submissions.json`.
2. Run this notebook (or `python Code/run_round.py --round N`).
3. Submit the printed query strings to the portal, one per function.
4. Commit the saved log in `Results/` to GitHub.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))   # run this notebook from the Code/ folder
import numpy as np
from bbo_core import (load_function_data, load_submissions, transform_y, fit_gp,
                      loo_check, propose_query, format_query)
from run_round import CONFIGS

ROUND = 1        # <- change to the round you are preparing
SEED = 42

## 1. Data

Each function's data = the official starter `.npy` files + every real portal result you have logged.

In [ ]:
submissions = load_submissions()
for fn in range(1, 9):
    X, y, n_init = load_function_data(fn, submissions=submissions)
    print(f"Function {fn}: {X.shape[1]}D, {n_init} starter + {len(y) - n_init} logged points, "
          f"best = {y.max():.6g} at {np.round(X[np.argmax(y)], 4)}")

## 2. Surrogate check (leave-one-out)

Before trusting a surrogate, refit it without each point and predict that point.
A calibration ratio near 1 means the model's uncertainty is roughly honest; well above 1 means it is overconfident.
With 10 to 40 points these numbers move a lot, so treat them as a rough guide.

In [ ]:
for fn in range(1, 9):
    X, y, _ = load_function_data(fn, submissions=submissions)
    cfg = CONFIGS[fn]
    mae, ratio = loo_check(X, transform_y(y, cfg['transform']), nu=cfg['nu'], noise=cfg['noise'])
    print(f"Function {fn}: LOO MAE = {mae:.4f}, calibration ratio = {ratio:.2f}")

## 3. Propose queries

For each function: fit the GP, generate global and local candidates, score them with the
function's acquisition setting (UCB or EI), and keep the best one. Every query is checked
against the portal range [0, 0.999999] before formatting.

In [ ]:
queries = {}
for fn in range(1, 9):
    X, y, _ = load_function_data(fn, submissions=submissions)
    q, diag = propose_query(X, y, CONFIGS[fn], seed=SEED + fn)
    queries[fn] = format_query(q)
    print(f"Function {fn} [{CONFIGS[fn]['acquisition'].upper()}]: {queries[fn]}")

## 4. Progress plot

Saved to `Results/progress.png`. Grey = starter data, red = your queries, blue = best so far.

In [ ]:
import subprocess
subprocess.run([sys.executable, 'plot_progress.py'], check=True)
from IPython.display import Image
Image(filename='../Results/progress.png')

## 5. Notes for this round

*Write 2 to 4 sentences here: what changed since last round, why, and anything surprising in the results.
These notes feed directly into the Model Card.*